# Process Wealth Inequality Data

**Purpose.** Harmonize World Inequality Database records with the study country names and derive inequality indicators.

**Inputs.** World Inequality Database export and country-name mappings.

**Outputs.** Country-year wealth-distribution indicators.

Run the code cells from top to bottom after updating the local paths described in the repository README. The notebook has no stored outputs; figures and tables are generated when it is executed.

In [ ]:
# Cell | Dependencies and country-name harmonization
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
region_mapping = {
    'North Africa': ['Algeria', 'Egypt', 'Libya', 'Mauritania', 'Morocco', 'Tunisia'],
    'West Africa': ['Benin', 'Burkina Faso', "Ivory Coast", 'Gambia', 'Ghana',
                    'Guinea', 'Guinea-Bissau', 'Liberia', 'Mali', 'Niger',
                    'Nigeria', 'Senegal', 'Sierra Leone', 'Togo'],
    'Central Africa': ['Cameroon', 'Central African Republic', 'Chad', 'Republic of the Congo',
                       'Democratic Republic of the Congo', 'Equatorial Guinea', 'Gabon'],
    'East Africa': ['Burundi', 'Djibouti', 'Eritrea', 'Ethiopia', 'Kenya',
                    'Rwanda', 'Somalia', 'South Sudan', 'Sudan', 'United Republic of Tanzania', 'Uganda'],
    'Southern Africa': ['Angola', 'Botswana', 'eSwatini', 'Lesotho', 'Madagascar', 'Malawi',
                        'Mozambique', 'Namibia', 'South Africa', 'Zambia', 'Zimbabwe']
}
country_to_region = {c: r for r, cs in region_mapping.items() for c in cs}
all_mapped_countries = list(country_to_region.keys())

# ============================================================================
# Harmonize WID country labels with the study-country names.
wid_country_mapping = {
    'Swaziland': 'eSwatini',
    'Congo': 'Republic of the Congo',
    'DR Congo': 'Democratic Republic of the Congo',
    'Tanzania': 'United Republic of Tanzania',
}

wid_country_mapping['Cote d' + chr(0x2019) + 'Ivoire'] = 'Ivory Coast'

for k, v in wid_country_mapping.items():
    print(f"  '{k}' -> '{v}'")

In [ ]:
# Cell | World Inequality Database ingestion
wid_data_path = r'E:\Africa_Infrastructure_Data\wid\WID_Data_01042026-091823.xls'

# Read the headerless WID data sheet and assign analysis fields.
df_wid_raw = pd.read_excel(wid_data_path, sheet_name='Data', header=None)

df_wid_raw.columns = ['country', 'variable_info', 'percentile', 'year', 'value']

print(df_wid_raw.columns.tolist())
print(df_wid_raw.head())
print(sorted(df_wid_raw['country'].unique())[:10])
print(df_wid_raw['percentile'].unique())

In [ ]:
# Cell | Country, region and numeric-value normalization
# ============================================================================

# Work on a copy to preserve the source table.
df_wid = df_wid_raw.copy()
df_wid['country'] = df_wid['country'].str.strip()

# Harmonize WID country labels with the study-country names.
df_wid['country_std'] = df_wid['country'].replace(wid_country_mapping)

df_wid = df_wid[df_wid['country_std'].isin(all_mapped_countries)].copy()

# Attach the regional classification to each country.
df_wid['region'] = df_wid['country_std'].map(country_to_region)

df_wid['value'] = pd.to_numeric(df_wid['value'], errors='coerce')
df_wid['year'] = pd.to_numeric(df_wid['year'], errors='coerce')

# Retain complete observations required for estimation.
df_wid = df_wid.dropna(subset=['value', 'year']).copy()
df_wid['year'] = df_wid['year'].astype(int)

present_countries = set(df_wid['country_std'].unique())
missing_countries = set(all_mapped_countries) - present_countries
if missing_countries:
    pass
else:
    pass



In [ ]:
# Cell | Wealth-share reshaping and inequality-ratio construction
# ============================================================================

# Work on a copy to preserve the source table.
df_inequality = df_wid[['country_std', 'region', 'year', 'percentile', 'value']].copy()

# Reshape the long table into indicator columns.
# Place wealth shares for each percentile group in separate columns.
df_pivot = df_inequality.pivot_table(
    index=['country_std', 'region', 'year'],
    columns='percentile',
    values='value',
    aggfunc='first'
).reset_index()

df_pivot.columns.name = None

# ============================================================================
# Compute top-to-bottom wealth-share ratios as inequality indicators.
df_pivot['top1_bottom50_ratio'] = df_pivot['p99p100'] / df_pivot['p0p50']

df_pivot['top10_bottom50_ratio'] = df_pivot['p90p100'] / df_pivot['p0p50']

df_pivot['top1_bottom50_ratio'] = df_pivot['top1_bottom50_ratio'].replace([np.inf, -np.inf], np.nan)
df_pivot['top10_bottom50_ratio'] = df_pivot['top10_bottom50_ratio'].replace([np.inf, -np.inf], np.nan)

print(df_pivot.head(10))
print(df_pivot.columns.tolist())
print(df_pivot[['p0p50', 'p90p100', 'p99p100', 'top1_bottom50_ratio', 'top10_bottom50_ratio']].describe())
print(df_pivot[df_pivot['year'] == 2024][['country_std', 'p0p50', 'p90p100', 'p99p100', 
                                            'top1_bottom50_ratio', 'top10_bottom50_ratio']].head(10))

In [ ]:
# Cell | Study-period filtering: 2015–2024
df_inequality_filtered = df_pivot[
    (df_pivot['year'] >= 2015) & (df_pivot['year'] <= 2024)
].copy()

print(df_inequality_filtered['year'].value_counts().sort_index())
for year in sorted(df_inequality_filtered['year'].unique()):
    n_countries = df_inequality_filtered[df_inequality_filtered['year'] == year]['country_std'].nunique()

print(df_inequality_filtered.head(20))

In [ ]:
# Cell | Processed inequality-table inspection
df_inequality_filtered

In [ ]:
# Cell | Export of the country-year inequality panel
output_path = r'E:\Africa_Infrastructure_Data\wealth_inequality_2015_2024.xlsx'
df_inequality_filtered.to_excel(output_path, index=False)